# Chapter 6 | Data Exercise 1 | **Online vs. Offline Prices in Another Country** #

*Use the same billion-prices dataset as in the case study. Pick another country, exclude sales and promotions, and test whether online and offline prices are the same, on average. Compare your results to the case study.* [*]

**Main question**: TODO: are online and offline prices the same, on average, in the country I picked?
**Chosen statistic**: TODO

## 1. Read the data ##

In [1]:
import warnings

import numpy as np
import pandas as pd
from scipy import stats
from plotnine import *

warnings.filterwarnings("ignore")

In [2]:
from bkda.paths import DATA_DIR, ROOT_DIR

HERE = f"{ROOT_DIR}chapters/ch06-testing_hypotheses/exercises/01-billion_prices_other_country/"
data_out = f"{HERE}data/clean/"
output = f"{HERE}output/"

In [3]:
from bkda.helpers import *

In [4]:
# Read online_offline_ALL_clean.csv (encoding="latin1", as in 6.A1)
data_in = f"{DATA_DIR}billion-prices/clean/"
data = pd.read_csv(data_in + "online_offline_ALL_clean.csv", encoding="latin1")


In [5]:
data.head()

,COUNTRY,retailer,retailer_s,date,day,month,year,id,price,price_online,...,DEVICEID,TIME,ZIPCODE,PHOTO,OTHERSKUITEM,COMMENTS,PRICETYPE,CODE,sale_online,country_s
0,ARGENTINA,1,ARGENTINA_1,2015-03-19,19.0,3.0,2015.0,201209030113,429.0,429.0,...,891df49fb1b12aa0,21:03,8300,20150319_210351.jpg,NaN,NaN,NaN,124816.0,NaN,Argentina
1,ARGENTINA,1,ARGENTINA_1,2015-03-19,19.0,3.0,2015.0,4710268235965,189.0,189.0,...,891df49fb1b12aa0,21:26,8300,20150319_212653.jpg,NaN,NaN,NaN,124816.0,NaN,Argentina
2,ARGENTINA,1,ARGENTINA_1,2015-03-19,19.0,3.0,2015.0,4905524916874,6999.0,6999.0,...,891df49fb1b12aa0,21:19,8300,20150319_211929.jpg,NaN,NaN,NaN,124816.0,NaN,Argentina
3,ARGENTINA,1,ARGENTINA_1,2015-03-19,19.0,3.0,2015.0,4905524925784,1999.0,2099.0,...,891df49fb1b12aa0,21:08,8300,20150319_210847.jpg,NaN,NaN,NaN,124816.0,NaN,Argentina
4,ARGENTINA,1,ARGENTINA_1,2015-03-19,19.0,3.0,2015.0,4905524931310,2899.0,2899.0,...,891df49fb1b12aa0,21:06,8300,20150319_210627.jpg,NaN,NaN,NaN,124816.0,NaN,Argentina


## 2. Pick a country and build the sample ##
> TODO:
> - Compare sample sizes across countries and justify the pick.
> - Check how `PRICETYPE` and `sale_online` are filled **in that country** before reusing the 6.A1 filter. What happens to the sample if you require `PRICETYPE == "Regular Price"`?
> - Look for extreme prices, as in 6.A1.

In [12]:
# Compare sample sizes across countries
data["COUNTRY"].value_counts().sort_values(ascending=False)

COUNTRY
USA            20193
CANADA          4422
AUSTRALIA       4190
ARGENTINA       4053
SOUTHAFRICA     3306
UK              2401
JAPAN           2296
BRAZIL          2077
GERMANY         1795
CHINA            520
Name: count, dtype: int64

Just by using samples size, Canada, Australia, and Argentina would be good choices. Countries such as China have less observations and can reduce the effectiveness of our research.

In [14]:
# Coverage of PRICETYPE and sale_online by country
# 1. What values exist at all?
for col in ["PRICETYPE", "sale_online"]:
    print(data[col].value_counts(dropna=False), "\n")

PRICETYPE
NaN                      34348
Regular Price             9496
Sale/Discounted Price     1409
Name: count, dtype: int64 

sale_online
NaN    41109
1.0     4144
Name: count, dtype: int64 



In [15]:
# Same counts per country
pd.crosstab(
    data["COUNTRY"],
    data["PRICETYPE"].fillna("missing"),
    margins=True,
)

PRICETYPE,Regular Price,Sale/Discounted Price,missing,All
COUNTRY,,,,
ARGENTINA,0,0,4053,4053
AUSTRALIA,0,0,4190,4190
BRAZIL,153,23,1901,2077
CANADA,0,0,4422,4422
CHINA,19,5,496,520
GERMANY,426,12,1357,1795
JAPAN,622,0,1674,2296
SOUTHAFRICA,542,17,2747,3306
UK,0,0,2401,2401


In [16]:
# Same thing, sale_online
pd.crosstab(
    data["COUNTRY"],
    data["sale_online"].fillna("missing"),
    margins=True,
)

sale_online,1.0,missing,All
COUNTRY,,,
ARGENTINA,0,4053,4053
AUSTRALIA,164,4026,4190
BRAZIL,46,2031,2077
CANADA,0,4422,4422
CHINA,0,520,520
GERMANY,9,1786,1795
JAPAN,0,2296,2296
SOUTHAFRICA,0,3306,3306
UK,140,2261,2401


In [17]:
# share missing per country
(
    data.groupby("COUNTRY")[["PRICETYPE", "sale_online"]]
    .agg(lambda s: s.isna().mean())
    .round(2)
    .sort_values("PRICETYPE")
)

,PRICETYPE,sale_online
COUNTRY,,
USA,0.55,0.81
JAPAN,0.73,1.00
GERMANY,0.76,0.99
SOUTHAFRICA,0.83,1.00
BRAZIL,0.92,0.98
CHINA,0.95,1.00
ARGENTINA,1.00,1.00
CANADA,1.00,1.00
AUSTRALIA,1.00,0.96


In [25]:
# quick check with value_counts
data.groupby("COUNTRY")["sale_online"].value_counts(dropna=False, normalize=True).unstack(fill_value=0).round(2)

sale_online,1.0,NaN
COUNTRY,,
ARGENTINA,0.00,1.00
AUSTRALIA,0.04,0.96
BRAZIL,0.02,0.98
CANADA,0.00,1.00
CHINA,0.00,1.00
GERMANY,0.01,0.99
JAPAN,0.00,1.00
SOUTHAFRICA,0.00,1.00
UK,0.06,0.94


In [23]:
# quick check with value_counts
data.groupby("COUNTRY")["PRICETYPE"].value_counts(dropna=False, normalize=True).unstack(fill_value=0).round(2)

PRICETYPE,Regular Price,Sale/Discounted Price,NaN
COUNTRY,,,
ARGENTINA,0.00,0.00,1.00
AUSTRALIA,0.00,0.00,1.00
BRAZIL,0.07,0.01,0.92
CANADA,0.00,0.00,1.00
CHINA,0.04,0.01,0.95
GERMANY,0.24,0.01,0.76
JAPAN,0.27,0.00,0.73
SOUTHAFRICA,0.16,0.01,0.83
UK,0.00,0.00,1.00


In [ ]:
# Filtered sample for the chosen country


## 3. Define the statistic ##
> TODO: Compute the online - offline difference, describe it, and plot its distribution. Remember the currency is not USD.

In [ ]:
# Price difference and summary statistics


In [ ]:
# Histogram of the difference


## 4. Hypotheses ##
> TODO: Write $H_0$ and $H_A$ in LaTeX (two-sided). Fix the significance level before testing.

## 5. Carry out the test ##
> TODO: SE and t-statistic by hand, then confirm with `stats.ttest_1samp`.

In [ ]:
# SE and t-statistic by hand


In [ ]:
# SciPy t-test and p-value


## 6. Confidence interval ##
> TODO: 95% CI for the average difference. Does it contain zero?

In [ ]:
# 95% CI


## 7. Compare with the case study (USA) ##
> TODO: Put both countries side by side (n, mean difference, SE, t, p, CI). Can you compare the mean differences directly when the currencies differ? What would make them comparable?

In [ ]:
# Comparison table


## 8. Conclusion and external validity ##
> TODO: Plain-language answer, and the population it applies to.